In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini 2.5 Flash Image（Nano Banana 🍌）在 Vertex AI 上的图像生成

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> 在 Colab 中打开
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fgetting-started%2Fintro_gemini_2_5_image_gen.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> 在 Colab Enterprise 中打开
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> 在 Vertex AI Workbench 中打开
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> 在 GitHub 上查看
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>分享至：</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/getting-started/intro_gemini_2_5_image_gen.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>

| 作者 |
| --- |
| [Katie Nguyen](https://github.com/katiemn) |

## 概述

Gemini 2.5 Flash Image 是一款强大的通用多模态模型，提供最先进的图像生成和对话式图像编辑功能。这使您能够与 Gemini 对话，并创建或编辑包含交织文本的图像。

在本教程中，您将学习如何在 Vertex AI 中使用 Google Gen AI SDK 调用 Gemini 2.5 Flash Image，尝试以下场景：
  - 图像生成：
    - 文本到图像生成
    - 图像与文本交织序列
  - 图像编辑：
    - 图像到图像（主体定制与风格迁移）
    - 多轮图像编辑与定位
    - 使用多张参考图像进行编辑

## 开始使用

### 安装 Python 版 Google Gen AI SDK

In [ ]:
%pip install --upgrade --quiet google-genai

### 验证您的 Notebook 环境（仅限 Colab）

如果您在 Google Colab 上运行此 notebook，请运行以下单元格以验证您的环境。

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### 导入库

In [ ]:
from io import BytesIO

from IPython.display import Image, Markdown, display
from PIL import Image as PIL_Image
from google import genai
from google.genai.types import FinishReason, GenerateContentConfig, ImageConfig, Part
import matplotlib.image as img
import matplotlib.pyplot as plt
import requests

### 设置 Google Cloud 项目信息并创建客户端

要开始使用 Vertex AI，您必须拥有一个现有的 Google Cloud 项目，并[启用 Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)。

了解有关[设置项目和开发环境](https://cloud.google.com/vertex-ai/docs/start/cloud-environment)的更多信息。

In [ ]:
import os

PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

### 加载图像模型


In [ ]:
MODEL_ID = "gemini-2.5-flash-image"

## 图像生成

首先，您将向 Gemini 2.5 Flash Image 发送文本提示，描述您希望生成的图像。


### 文本到图像

在下面的单元格中，您将调用 `generate_content` 方法，并传入以下参数：

  - `model`：您要使用的模型 ID。
  - `contents`：您的提示，在本例中是描述要生成图像的纯文本用户消息。
  - `config`：用于指定内容设置的配置。
    - `response_modalities`：要生成图像，您必须在 `response_modalities` 列表中包含 `IMAGE`。
    - `ImageConfig`：设置 `aspect_ratio`（宽高比）。有效比例为："1:1"、"3:2"、"2:3"、"3:4"、"4:3"、"4:5"、"5:4"、"9:16"、"16:9"、"21:9"
    - `candidate_count`：要生成的候选数量。


所有生成的图像均包含 [SynthID 水印](https://deepmind.google/technologies/synthid/)，可通过 [Vertex AI Studio](https://cloud.google.com/generative-ai-studio?hl=en) 中的 Media Studio 进行验证。

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="a cartoon infographic on flying sneakers",
    config=GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="9:16",
        ),
        candidate_count=1,
    ),
)

# Check for errors if an image is not generated
if response.candidates[0].finish_reason != FinishReason.STOP:
    reason = response.candidates[0].finish_reason
    raise ValueError(f"Prompt Content Error: {reason}")


for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

### 文本到图像与文本

除了生成图像外，Gemini 还可以创建图像与文本交织的序列。

例如，您可以要求模型生成一份香蕉面包食谱，并附上展示烹饪过程不同阶段的图像。或者，您可以要求模型生成不同野花的图像，并附上相应的标题和描述。

让我们通过提示 Gemini 2.5 Flash Image 创建一份花生酱和果冻三明治的制作教程，来尝试交织文本和图像的功能。

您会注意到，在提示中我们要求模型为每个步骤生成文本和图像。这会促使模型创建交织的文本和图像。此外，请确保设置 `response_modalities=["TEXT", "IMAGE"]`，以便在响应中同时获取两种模态。

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="Create a tutorial explaining how to make a peanut butter and jelly sandwich in three easy steps. For each step, provide a title with the number of the step, an explanation, and also generate an image to illustrate the content. Label each image with the step number but no other words.",
    config=GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="4:3",
        ),
    ),
)

for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

## 图像编辑

Gemini 2.5 Flash Image 可以从多张参考图像生成图像到图像的输出。这对于确保角色一致性、生成徽标、迁移风格以及插入或删除对象等任务非常有用。

### 主体定制

让我们尝试一个主体定制示例，要求 Gemini 2.5 Flash Image 创建一张该狗的铅笔素描风格图像。


#### 下载狗的图像

以下示例使用来自 Cloud Storage 的图像。如果您希望使用不同的图像，可以更改 `wget` 命令中的 URL，或者如果您有本地文件，请在后续步骤中更新 `subject_image` 变量。

In [ ]:
!wget https://storage.googleapis.com/cloud-samples-data/generative-ai/image/dog-1.jpg

In [ ]:
subject_image = "dog-1.jpg"  # @param {type: 'string'}

# Display the image
fig, axis = plt.subplots(1, 1, figsize=(6, 12))
axis.imshow(img.imread(subject_image))
axis.axis("off")
plt.show()

#### 发送请求

由于您在请求中包含来自本地图像的数据，因此需要在请求的 `contents` 中包含 `Part.from_bytes`。

In [ ]:
with open(subject_image, "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_bytes(
            data=image,
            mime_type="image/jpeg",
        ),
        "Create a pencil sketch image of this dog wearing a cowboy hat in a western-themed setting.",
    ],
    config=GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="1:1",
        ),
        candidate_count=1,
    ),
)

for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

### 风格迁移

在下一个示例中，您将使用客厅的风格来重新构想同一风格的厨房。

#### 下载客厅图像

同样，以下示例使用来自 Cloud Storage 的图像。如果您希望使用不同的图像，可以更改 `wget` 命令中的 URL，或者如果您有本地文件，请在后续步骤中更新 `style_image` 变量。

In [ ]:
!wget https://storage.googleapis.com/cloud-samples-data/generative-ai/image/living-room.png

In [ ]:
style_image = "living-room.png"  # @param {type: 'string'}

# Display the image
fig, axis = plt.subplots(1, 1, figsize=(6, 12))
axis.imshow(img.imread(style_image))
axis.axis("off")
plt.show()

#### 发送请求

In [ ]:
with open(style_image, "rb") as f:
    image = f.read()

response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_bytes(
            data=image,
            mime_type="image/png",
        ),
        "Using the concepts, colors, and themes from this living room generate a kitchen and dining room with the same aesthetic.",
    ],
    config=GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="21:9",
        ),
        candidate_count=1,
    ),
)

for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

### 多轮图像编辑

在下一节中，您将提供一张起始图像，并通过与 Gemini 2.5 Flash Image 聊天，迭代地修改图像的某些方面。


在下一个示例中，您将使用存储在 Google Cloud Storage 中的图像，而不是本地图像。运行下一步以查看香水瓶的起始图像。如果您想使用 Cloud Storage 中的不同图像，请替换下面的 `perfume_url` 和 `perfume_uri`。

In [ ]:
perfume_url = (
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/perfume.jpg"
)
perfume_uri = "gs://cloud-samples-data/generative-ai/image/perfume.jpg"


# Display the image
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
perfume_image = PIL_Image.open(BytesIO(requests.get(perfume_url).content))
axes[0].imshow(perfume_image)
for i, ax in enumerate(axes):
    ax.axis("off")
plt.show()

#### 开始聊天

在下一步中，您将启动一个聊天，以便通过与 Gemini 对话来持续编辑图像。由于您现在使用的是存储在 Cloud Storage 中的参考图像，因此将在 `message` 内容中使用 `Part.from_uri`。

In [ ]:
chat = client.chats.create(model=MODEL_ID)

response = chat.send_message(
    message=[
        Part.from_uri(
            file_uri=perfume_uri,
            mime_type="image/jpeg",
        ),
        "change the perfume color to a light purple",
    ],
    config=GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="3:2",
        ),
    ),
)

data = perfume_uri
for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))
        data = part.inline_data.data

现在，您将在现有聊天中的新消息里包含之前的图像数据，以及一个新的文本提示，来更新之前生成的图像。这次，您将要求在香水瓶上写一个单词，由于 Gemini 能够处理不同语言，让我们要求用法语写这个单词。

In [ ]:
response = chat.send_message(
    message=[
        Part.from_bytes(
            data=data,
            mime_type="image/jpeg",
        ),
        "inscribe the word flowers in French on the perfume bottle in a delicate white cursive font",
    ],
    config=GenerateContentConfig(
        response_modalities=["IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="3:2",
        ),
    ),
)

for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))

### 多张参考图像

使用 Gemini 2.5 Flash Image 编辑图像时，您还可以提供多张输入图像来创建新图像。在下一个示例中，您将向 Gemini 提供一张女性图像和一张手提箱图像。然后，您将要求 Gemini 将这些图像中的对象组合起来，创建一张新图像。您还将要求 Gemini 提供配套文字。


运行以下单元格以可视化存储在 Cloud Storage 中的起始图像。

In [ ]:
person_url = (
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/woman.jpg"
)
suitcase_url = (
    "https://storage.googleapis.com/cloud-samples-data/generative-ai/image/suitcase.png"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
person_image = PIL_Image.open(BytesIO(requests.get(person_url).content))
axes[0].imshow(person_image)
suitcase_image = PIL_Image.open(BytesIO(requests.get(suitcase_url).content))
axes[1].imshow(suitcase_image)
for i, ax in enumerate(axes):
    ax.axis("off")
plt.show()

现在，您将发送请求。与之前图像编辑调用的唯一区别是，您将根据参考图像的数量提供多个 `Part.from_uri` 实例。

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/image/suitcase.png",
            mime_type="image/png",
        ),
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/image/woman.jpg",
            mime_type="image/jpeg",
        ),
        "Generate an image of the woman pulling the suitcase in an airport. Separately, write a short caption for this image that would be suitable for a social media post.",
    ],
    config=GenerateContentConfig(
        response_modalities=["TEXT", "IMAGE"],
        image_config=ImageConfig(
            aspect_ratio="9:16",
        ),
        candidate_count=1,
    ),
)


for part in response.candidates[0].content.parts:
    if part.text:
        display(Markdown(part.text))
    if part.inline_data:
        display(Image(data=part.inline_data.data, width=500))